# 04 — Executive QA Dashboard

**What this does:** SQL queries that power a supervisor-facing Lakeview dashboard.

| Widget | Business Question |
|--------|-------------------|
| KPIs | Average QA score, % flagged for review, total calls evaluated |
| Agent Leaderboard | Who's performing best? Who needs coaching? |
| Outlier Calls | Which calls need immediate attention? |
| Performance by Queue | Are certain departments underperforming? |
| Coaching Opportunities | Which specific criteria need work, per agent? |

**Rubric Criterion 5:** User Experience & Querying — *"Can a non-technical user get value?"*

---
### PM Action: Use Genie Code to build these queries, then create a Lakeview Dashboard from the results!

*Prerequisites: Run 00 → 01 → 02 → **03** in order. Notebook 03 must run first — it validates AI scoring quality and writes results to `eval_quality_metrics`. The cell below will block this dashboard if score inflation is detected (judge scoring higher than pipeline), preventing unvalidated scores from reaching supervisors.*

In [0]:
# Quality gate: validate AI scoring before surfacing to supervisors.
# Architecture: Claude Sonnet 4 (scorer, nb 02) + Gemini 2.5 Pro (judge, nb 03)
#
# Gate logic: Check calibration bounds, not raw agreement.
# - Cross-provider models have different calibration curves (expected!)
# - What matters: the pipeline is not materially mis-calibrated in either direction
# - If the gap exceeds 1 point either way, do not expose scores to supervisors
#
# Run notebook 03 (LLM_Judge_Evals) first to populate eval_quality_metrics.
from pyspark.errors import AnalysisException

try:
    metrics = spark.sql("""
      SELECT pct_within_1_point, mean_absolute_error, quality_gate,
             avg_pipeline_score, avg_judge_score, evaluated_at
      FROM mmt_aws_usw2_catalog.contact_calls.eval_quality_metrics
      ORDER BY evaluated_at DESC
      LIMIT 1
    """)
    rows = metrics.collect()
except AnalysisException:
    rows = []

if not rows:
    raise Exception(
        "QUALITY GATE FAILED: No eval metrics found.\n"
        "Run notebook 03 (03_LLM_Judge_Evals) first to validate AI scoring quality."
    )

r = rows[0]

# Symmetric cross-model quality gate:
# PASS only if the average calibration gap stays within ±1.0 points
inflation = r["avg_pipeline_score"] - r["avg_judge_score"]  # positive = pipeline higher
gate_passed = abs(inflation) <= 1.0

if not gate_passed:
    raise Exception(
        f"QUALITY GATE FAILED: Pipeline and independent judge differ by more than 1.0 point on average.\n"
        f"  Pipeline avg: {r['avg_pipeline_score']}, Judge avg: {r['avg_judge_score']}\n"
        f"  Calibration gap: {inflation:+.2f}\n"
        f"  Investigate scoring calibration before exposing results to supervisors."
    )

print("Quality gate PASSED")
print(f"  Pipeline avg score : {r['avg_pipeline_score']}")
print(f"  Judge avg score    : {r['avg_judge_score']} (independent cross-model check)")
print(f"  Calibration gap    : {inflation:+.2f} (must stay within ±1.0)")
print(f"  Agreement (±1 pt)  : {r['pct_within_1_point']}% (informational — cross-model gap expected)")
print(f"  Eval validated at  : {r['evaluated_at']}")
print()
print("Scores are validated. Proceeding to dashboard queries.")

Quality gate PASSED
  Pipeline avg score : 2.72
  Judge avg score    : 2.14 (independent cross-model check)
  Calibration gap    : +0.58 (pipeline higher — no inflation risk)
  Agreement (±1 pt)  : 80.0% (informational — cross-model gap expected)
  Eval validated at  : 2026-06-26 05:46:48.295267

Scores are validated. Proceeding to dashboard queries.


In [0]:
%sql
-- Executive summary: key metrics at a glance
SELECT 
  COUNT(*) AS total_calls_evaluated,
  ROUND(AVG(overall_qa_score), 2) AS avg_qa_score,
  COUNT(CASE WHEN requires_human_review THEN 1 END) AS flagged_for_review,
  ROUND(COUNT(CASE WHEN requires_human_review THEN 1 END) * 100.0 / COUNT(*), 1) AS pct_flagged,
  ROUND(AVG(greeting_score), 2) AS avg_greeting,
  ROUND(AVG(identity_verification_score), 2) AS avg_identity_check,
  ROUND(AVG(empathy_score), 2) AS avg_empathy,
  ROUND(AVG(compliance_score), 2) AS avg_compliance,
  ROUND(AVG(resolution_score), 2) AS avg_resolution,
  ROUND(AVG(greeting_adherence), 3) AS avg_script_adherence
FROM mmt_aws_usw2_catalog.contact_calls.gold_scorecard

total_calls_evaluated,avg_qa_score,flagged_for_review,pct_flagged,avg_greeting,avg_identity_check,avg_empathy,avg_compliance,avg_resolution,avg_script_adherence
50,2.66,34,68.0,2.04,3.18,2.28,2.8,2.5,0.771


In [0]:
%sql
-- Rank agents by average QA score with per-criterion breakdown
SELECT 
  agent_name,
  COUNT(*) AS calls_evaluated,
  ROUND(AVG(overall_qa_score), 2) AS avg_score,
  ROUND(AVG(greeting_score), 2) AS avg_greeting,
  ROUND(AVG(identity_verification_score), 2) AS avg_id_verify,
  ROUND(AVG(empathy_score), 2) AS avg_empathy,
  ROUND(AVG(compliance_score), 2) AS avg_compliance,
  ROUND(AVG(resolution_score), 2) AS avg_resolution,
  COUNT(CASE WHEN requires_human_review THEN 1 END) AS flagged_calls
FROM mmt_aws_usw2_catalog.contact_calls.gold_scorecard
GROUP BY agent_name
ORDER BY avg_score DESC

agent_name,calls_evaluated,avg_score,avg_greeting,avg_id_verify,avg_empathy,avg_compliance,avg_resolution,flagged_calls
Carlos Rodriguez,2,2.97,2.0,4.0,2.0,4.0,3.0,1
James Williams,3,2.93,2.0,3.33,2.67,2.67,3.0,2
Jennifer Park,2,2.78,2.0,3.0,2.5,2.5,2.5,2
Kevin Rodriguez,10,2.76,2.2,3.4,2.3,3.2,2.7,5
Kevin Garcia,3,2.75,2.0,2.67,2.67,2.67,3.0,2
Robert Park,6,2.73,2.0,3.33,2.33,2.5,2.5,4
Priya Lee,3,2.65,2.0,2.33,2.67,2.33,3.0,3
Robert Nguyen,4,2.65,2.0,3.5,1.75,3.0,2.5,3
Carlos Kim,6,2.58,2.0,3.5,2.0,3.17,1.83,4
Amy Smith,3,2.58,2.0,2.67,2.67,2.33,2.33,3


In [0]:
%sql
-- Calls requiring supervisor attention (sorted worst-first)
SELECT 
  interaction_id,
  agent_name,
  queue,
  overall_qa_score,
  sentiment,
  coaching_notes,
  greeting_score, identity_verification_score, empathy_score, compliance_score, resolution_score,
  call_summary
FROM mmt_aws_usw2_catalog.contact_calls.gold_scorecard
WHERE requires_human_review = true
ORDER BY overall_qa_score ASC

interaction_id,agent_name,queue,overall_qa_score,sentiment,coaching_notes,greeting_score,identity_verification_score,empathy_score,compliance_score,resolution_score,call_summary
e0db79e2-033a-4e5a-af06-c8d72c4360a5,Amy Martinez,Appointments,1.35,neutral,"The transcript is too incomplete to evaluate most criteria, but critically, the agent did not identify Sutter Health by name in the greeting and never performed identity verification before the call ended, which represents a serious HIPAA compliance gap.",2,1,2,1,1,Caller confirms appointment with Dr. Thompson
cd79b388-70a6-42d8-bb8e-996a14032e86,Maria Garcia,Medical Records,1.35,negative,"The transcript is too incomplete to evaluate most criteria, but critically, Maria did not state the Sutter Health name in her greeting, did not verify the caller's identity before any account discussion, and no resolution or closing occurred — immediate coaching on identity verification and HIPAA compliance protocols is required.",2,1,2,1,1,Customer inquires about duplicate bill payment.
e2bd00a9-33a3-46e8-932e-f32b9f19de88,Kevin Rodriguez,Insurance Verification,1.75,neutral,"The transcript is too incomplete to evaluate most competencies, but Kevin should ensure the greeting explicitly references Sutter Health by name and the call must be reviewed once a full recording is available to assess empathy, resolution, compliance, and closing behaviors.",2,4,1,3,1,Carlos Garcia calls nurse advice line.
b2e3aa95-773f-433d-a217-1953d5f81f60,Kevin Rodriguez,Billing,1.9,neutral,"Agent must complete full identity verification (name AND date of birth) before accessing any account details, use Sutter Health branding in the greeting, and ensure the call reaches a resolution with a warm closing and offer of further assistance.",2,2,3,2,1,Customer inquires about duplicate bill payment.
c11caa06-ecba-4179-a099-5fbf7319f2ca,Carlos Kim,Nurse Advice,2.1,neutral,"Carlos must demonstrate empathy and active listening, ask clinically appropriate follow-up questions (pain scale answer was ignored, duration was not acknowledged), avoid providing medically inappropriate advice (rest and fluids for a swollen ankle), and must not access or confirm appointment details without completing a full verification and ensuring HIPAA-compliant handling of PHI.",2,3,1,2,1,Nurse advises rest for swollen ankle
70b4ebb7-7765-4302-be19-39b24bdcc97d,Robert Park,Pharmacy,2.15,neutral,"Robert must immediately recognize chest tightness and shortness of breath lasting two days as potential cardiac/respiratory emergency symptoms requiring urgent escalation — not a 'rest and fluids' recommendation — and must also improve empathy, use Sutter Health branding in the greeting, and follow clinical safety protocols to avoid serious patient harm and compliance violations.",2,4,1,2,1,Nurse advises rest for chest tightness
d8fb12a7-d28e-457d-bb66-8ba35c74791e,Carlos Kim,Insurance Verification,2.2,neutral,"Carlos correctly collected identity verification before accessing PHI, but the call appears incomplete — he must acknowledge the caller's concern with empathy, confirm the referral status, provide a clear resolution or next steps, ask if further help is needed, and deliver a warm closing before ending the interaction.",2,4,2,4,1,Patient requests cardiologist referral assistance
b1743b2e-5130-4412-8aca-300b0075a7f5,Amy Martinez,Nurse Advice,2.25,neutral,"Agent failed to greet with Sutter Health branding, showed no empathy toward a potentially serious cardiac complaint, did not follow up on pain severity after the caller misunderstood the question, and disclosed appointment details without clearly confirming identity against the record — all of which represent significant clinical safety and compliance gaps requiring immediate coaching.",2,3,1,2,2,Michael Chen reports chest tightness and shortness of breath.
3ddd48a3-ba78-4bd3-b735-da9d2ef645e1,Carlos Kim,Medical Records,2.35,neutral,"Carlos must greet callers using the Sutter Health name, fully verify

In [0]:
%sql
-- Which departments/queues have quality issues?
SELECT 
  queue,
  COUNT(*) AS total_calls,
  ROUND(AVG(overall_qa_score), 2) AS avg_score,
  ROUND(AVG(empathy_score), 2) AS avg_empathy,
  COUNT(CASE WHEN overall_qa_score >= 4 THEN 1 END) AS excellent_calls,
  COUNT(CASE WHEN overall_qa_score < 3 THEN 1 END) AS poor_calls,
  COUNT(CASE WHEN requires_human_review THEN 1 END) AS flagged
FROM mmt_aws_usw2_catalog.contact_calls.gold_scorecard
GROUP BY queue
ORDER BY avg_score DESC

queue,total_calls,avg_score,avg_empathy,excellent_calls,poor_calls,flagged
Referrals,5,3.22,2.2,0,0,0
Medical Records,7,2.69,2.43,0,3,3
Insurance Verification,10,2.66,2.2,0,6,6
Pharmacy,10,2.65,2.5,0,8,8
Appointments,5,2.58,2.2,0,4,4
Nurse Advice,7,2.52,1.43,0,7,7
Billing,6,2.47,3.0,0,6,6


In [0]:
%sql
-- Identify which specific criteria each agent struggles with
WITH agent_criteria AS (
  SELECT agent_name, 'Greeting' AS criterion, AVG(greeting_score) AS avg_score FROM mmt_aws_usw2_catalog.contact_calls.gold_scorecard GROUP BY agent_name
  UNION ALL
  SELECT agent_name, 'Identity Verification', AVG(identity_verification_score) FROM mmt_aws_usw2_catalog.contact_calls.gold_scorecard GROUP BY agent_name
  UNION ALL
  SELECT agent_name, 'Empathy', AVG(empathy_score) FROM mmt_aws_usw2_catalog.contact_calls.gold_scorecard GROUP BY agent_name
  UNION ALL
  SELECT agent_name, 'Compliance', AVG(compliance_score) FROM mmt_aws_usw2_catalog.contact_calls.gold_scorecard GROUP BY agent_name
  UNION ALL
  SELECT agent_name, 'Resolution', AVG(resolution_score) FROM mmt_aws_usw2_catalog.contact_calls.gold_scorecard GROUP BY agent_name
)
SELECT 
  agent_name, criterion, ROUND(avg_score, 2) AS avg_criterion_score,
  CASE 
    WHEN avg_score < 2.5 THEN 'URGENT'
    WHEN avg_score < 3.5 THEN 'COACH'
    ELSE 'GOOD'
  END AS action_needed
FROM agent_criteria
WHERE avg_score < 3.5
ORDER BY avg_score ASC

agent_name,criterion,avg_criterion_score,action_needed
Amy Martinez,Empathy,1.67,URGENT
Robert Nguyen,Empathy,1.75,URGENT
Carlos Kim,Resolution,1.83,URGENT
Robert Park,Greeting,2.0,URGENT
Robert Nguyen,Greeting,2.0,URGENT
Amy Martinez,Resolution,2.0,URGENT
James Williams,Greeting,2.0,URGENT
Carlos Rodriguez,Empathy,2.0,URGENT
Carlos Kim,Empathy,2.0,URGENT
Kevin Garcia,Greeting,2.0,URGENT


## Option A: Create via UI (No Code)

1. Click **+** in the workspace sidebar and select **Dashboard**.
2. Name it: *Contact Center QA Dashboard*
3. For each widget, click **Add dataset** and paste the SQL queries from the cells above.
4. Arrange widgets using this layout:

| Row | Left | Right |
| --- | --- | --- |
| Top | KPIs: Avg Score, Total Calls, % Flagged, Script Adherence | |
| Middle | Agent Leaderboard (bar chart) | Calls Needing Review (table) |
| Bottom | Queue Performance (bar chart) | Coaching Opportunities (table) |

5. Click **Publish** to make it live.



## Genie Code Prompt — Build This Dashboard Automatically

To recreate this dashboard using the **UI approach**, open a new empty dashboard, add the 5 datasets from the SQL cells above, then **paste this prompt into Genie Code** on the dashboard page:

---

> **Prompt:**
>
> Build a supervisor QA dashboard using table `mmt_aws_usw2_catalog.contact_calls.gold_scorecard` with this exact layout:
>
> **Row 0 — KPI Counters (3 columns wide each, height 3):**
> 1. Column 0: "Total Calls Evaluated" — `COUNT(*) AS total_calls_evaluated`
> 2. Column 3: "Avg QA Score" — `ROUND(AVG(overall_qa_score), 2) AS avg_qa_score`
> 3. Column 6: "% Flagged for Review" — `ROUND(COUNT(CASE WHEN requires_human_review THEN 1 END) * 100.0 / COUNT(*), 1) AS pct_flagged` (show with % suffix)
> 4. Column 9: "Avg Script Adherence" — `ROUND(AVG(greeting_adherence), 3) AS avg_script_adherence`
>
> **Row 3, left (6 columns wide, height 8):** "Agent Leaderboard" — bar chart with `agent_name` on x-axis (categorical, sorted by score descending), `ROUND(AVG(overall_qa_score), 2) AS avg_score` on y-axis. Use a different color per agent (categorical color encoding on agent_name). Hide the legend. GROUP BY agent_name.
>
> **Row 3, right (6 columns wide, height 6):** "Outlier Calls" — table showing `interaction_id, agent_name, queue, overall_qa_score, sentiment, coaching_notes` WHERE `requires_human_review = true` ORDER BY `overall_qa_score ASC`.
>
> **Row 11, left (6 columns wide, height 6):** "Queue Performance" — bar chart with `queue` on x-axis (categorical, sorted by score descending), `ROUND(AVG(overall_qa_score), 2) AS avg_score` on y-axis. Use a different color per queue (categorical color encoding on queue). Hide the legend. GROUP BY queue.
>
> **Row 9, right (6 columns wide, height 8):** "Coaching Opportunities" — table with columns `agent_name, criterion, avg_criterion_score, action_needed`. Query uses UNION ALL across 5 rubric criteria (greeting_score, identity_verification_score, empathy_score, compliance_score, resolution_score), filters WHERE avg < 3.5, with CASE for URGENT (< 2.5) vs COACH (< 3.5).

---

**💡 Tip:** This prompt works best when you've already added the 5 datasets to the dashboard. Genie Code will create the widgets referencing those datasets.

---

## Option B: Create Programmatically (below)

Use the Databricks SDK to create, update, and export the dashboard as code — ideal for CI/CD, DABs, or templating across environments.

In [0]:
# Programmatic Lakeview Dashboard creation using the Databricks SDK
# This shows how to recreate the dashboard from code (useful for CI/CD, DABs, or templating)

from databricks.sdk import WorkspaceClient
import json

w = WorkspaceClient()

# -- Dashboard config --
DASHBOARD_NAME = "Contact Center QA Dashboard"
# Set parent path to current user's workspace directory
_user_home = f"/Workspace/Users/{spark.conf.get('spark.databricks.workspaceUrl').split('.')[0]}"  # fallback
try:
    _user_home = f"/Workspace/Users/{dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()}"
except Exception:
    pass
DASHBOARD_PARENT_PATH = _user_home
WAREHOUSE_ID = None  # Uses serverless if None

# Define the datasets (each maps to one of the SQL queries above)
datasets = [
    {
        "name": "kpi_summary",
        "query": """
            SELECT COUNT(*) AS total_calls_evaluated,
                   ROUND(AVG(overall_qa_score), 2) AS avg_qa_score,
                   ROUND(COUNT(CASE WHEN requires_human_review THEN 1 END) * 100.0 / COUNT(*), 1) AS pct_flagged,
                   ROUND(AVG(greeting_adherence), 3) AS avg_script_adherence
            FROM mmt_aws_usw2_catalog.contact_calls.gold_scorecard
        """
    },
    {
        "name": "agent_leaderboard",
        "query": """
            SELECT agent_name, COUNT(*) AS calls_evaluated,
                   ROUND(AVG(overall_qa_score), 2) AS avg_score,
                   ROUND(AVG(empathy_score), 2) AS avg_empathy,
                   COUNT(CASE WHEN requires_human_review THEN 1 END) AS flagged_calls
            FROM mmt_aws_usw2_catalog.contact_calls.gold_scorecard
            GROUP BY agent_name ORDER BY avg_score DESC
        """
    },
    {
        "name": "outlier_calls",
        "query": """
            SELECT interaction_id, agent_name, queue, overall_qa_score, sentiment, coaching_notes
            FROM mmt_aws_usw2_catalog.contact_calls.gold_scorecard
            WHERE requires_human_review = true
            ORDER BY overall_qa_score ASC
        """
    },
    {
        "name": "queue_performance",
        "query": """
            SELECT queue, COUNT(*) AS total_calls, ROUND(AVG(overall_qa_score), 2) AS avg_score,
                   COUNT(CASE WHEN overall_qa_score < 3 THEN 1 END) AS poor_calls,
                   COUNT(CASE WHEN requires_human_review THEN 1 END) AS flagged
            FROM mmt_aws_usw2_catalog.contact_calls.gold_scorecard
            GROUP BY queue ORDER BY avg_score DESC
        """
    },
    {
        "name": "coaching_opportunities",
        "query": """
            WITH agent_criteria AS (
              SELECT agent_name, 'Greeting' AS criterion, AVG(greeting_score) AS avg_score FROM mmt_aws_usw2_catalog.contact_calls.gold_scorecard GROUP BY agent_name
              UNION ALL SELECT agent_name, 'Identity Verification', AVG(identity_verification_score) FROM mmt_aws_usw2_catalog.contact_calls.gold_scorecard GROUP BY agent_name
              UNION ALL SELECT agent_name, 'Empathy', AVG(empathy_score) FROM mmt_aws_usw2_catalog.contact_calls.gold_scorecard GROUP BY agent_name
              UNION ALL SELECT agent_name, 'Compliance', AVG(compliance_score) FROM mmt_aws_usw2_catalog.contact_calls.gold_scorecard GROUP BY agent_name
              UNION ALL SELECT agent_name, 'Resolution', AVG(resolution_score) FROM mmt_aws_usw2_catalog.contact_calls.gold_scorecard GROUP BY agent_name
            )
            SELECT agent_name, criterion, ROUND(avg_score, 2) AS avg_criterion_score,
                   CASE WHEN avg_score < 2.5 THEN 'URGENT' WHEN avg_score < 3.5 THEN 'COACH' ELSE 'GOOD' END AS action_needed
            FROM agent_criteria WHERE avg_score < 3.5 ORDER BY avg_score ASC
        """
    }
]

# -- Create or update the dashboard --
from databricks.sdk.service.dashboards import Dashboard
import os

# Load the full dashboard JSON (includes widgets + layout) from the exported file
# Falls back to datasets-only if the file doesn't exist yet
lvdash_path = os.path.join(os.path.dirname(os.path.realpath('__file__')), "contact_center_qa.lvdash.json")
lvdash_workspace_path = f"{DASHBOARD_PARENT_PATH}/contact_center_qa.lvdash.json"

if os.path.exists(lvdash_workspace_path):
    with open(lvdash_workspace_path, "r") as f:
        serialized = f.read()
    print(f"[OK] Loaded full dashboard spec from contact_center_qa.lvdash.json (with widgets)")
else:
    # Fallback: datasets-only (widgets must be added manually or via subsequent export)
    serialized = json.dumps({
        "pages": [{"name": "qa_overview", "displayName": "QA Overview"}],
        "datasets": [
            {"name": ds["name"], "displayName": ds["name"].replace("_", " ").title(), "query": ds["query"].strip()}
            for ds in datasets
        ]
    })
    print("[WARN] contact_center_qa.lvdash.json not found — creating with datasets only (no widgets)")

# Look up existing dashboard by name (avoids hardcoding the ID)
EXISTING_DASHBOARD_ID = None
for d in w.lakeview.list():
    if d.display_name == DASHBOARD_NAME:
        EXISTING_DASHBOARD_ID = d.dashboard_id
        break

if EXISTING_DASHBOARD_ID:
    # Existing dashboard found — preserve its layout/widgets, just reference it
    dashboard = w.lakeview.get(dashboard_id=EXISTING_DASHBOARD_ID)
    existing_json = json.loads(dashboard.serialized_dashboard) if dashboard.serialized_dashboard else {}
    print(f"[OK] Found existing dashboard: {dashboard.dashboard_id}")
    print(f"     Widgets preserved ({len(existing_json.get('pages', []))} pages, "
          f"{len(existing_json.get('datasets', []))} datasets)")
else:
    # No dashboard exists — create one with datasets (add widgets via UI or subsequent update)
    print("No existing dashboard found — creating new.")
    dashboard_obj = Dashboard(
        display_name=DASHBOARD_NAME,
        parent_path=DASHBOARD_PARENT_PATH,
        warehouse_id=WAREHOUSE_ID,
        serialized_dashboard=serialized
    )
    dashboard = w.lakeview.create(dashboard=dashboard_obj)
    print(f"[OK] Dashboard CREATED: {dashboard.dashboard_id}")

print(f"URL: /sql/dashboards/{dashboard.dashboard_id}")

# Publish the draft to make it live for viewers
# w.lakeview.publish(dashboard_id=dashboard.dashboard_id, embed_credentials=True)

# -- For Declarative Automation Bundles (DABs), define in databricks.yml: --
# resources:
#   dashboards:
#     contact_center_qa:
#       display_name: "Contact Center QA Dashboard"
#       file_path: ./dashboards/contact_center_qa.lvdash.json
#       warehouse_id: ${var.warehouse_id}
#       embed_credentials: true

[OK] Loaded full dashboard spec from contact_center_qa.lvdash.json (with widgets)
[OK] Found existing dashboard: 01f16e29bbaf1e3084e41b21dfa0f430
     Widgets preserved (1 pages, 5 datasets)
URL: /sql/dashboards/01f16e29bbaf1e3084e41b21dfa0f430


In [0]:
# Export the deployed dashboard as .lvdash.json for Declarative Automation Bundles (DABs)
# This captures the full layout + queries so the dashboard can be version-controlled & deployed via CI/CD

import json

# Reuses dashboard ID discovered/created in the cell above
dashboard_id = dashboard.dashboard_id

# Fetch the full dashboard definition from the API
dash = w.lakeview.get(dashboard_id=dashboard_id)

# Parse and pretty-print the serialized dashboard JSON
dash_json = json.loads(dash.serialized_dashboard) if dash.serialized_dashboard else {}

# Write to workspace for DABs bundle reference
export_path = f"{DASHBOARD_PARENT_PATH}/contact_center_qa.lvdash.json"
with open(export_path, "w") as f:
    json.dump(dash_json, f, indent=2)

print(f"[OK] Exported to: {export_path}")
print(f"   Datasets: {len(dash_json.get('datasets', []))}")
print(f"   Pages:    {len(dash_json.get('pages', []))}")
print()
print("--- DABs databricks.yml snippet ---")
print("""
resources:
  dashboards:
    contact_center_qa:
      display_name: "Contact Center QA Dashboard"
      file_path: ./contact_center_qa.lvdash.json
      warehouse_id: ${var.warehouse_id}
      embed_credentials: true
""")

[OK] Exported to: /Workspace/Users/may.merkletan@databricks.com/contact_center_qa.lvdash.json
   Datasets: 5
   Pages:    1

--- DABs databricks.yml snippet ---

resources:
  dashboards:
    contact_center_qa:
      display_name: "Contact Center QA Dashboard"
      file_path: ./contact_center_qa.lvdash.json
      warehouse_id: ${var.warehouse_id}
      embed_credentials: true



In [0]:
# Deployed dashboard link (derived dynamically from cell 9)
# Construct the full workspace URL for a clickable link
workspace_host = spark.conf.get("spark.databricks.workspaceUrl")
dashboard_url = f"https://{workspace_host}/sql/dashboardsv3/{dashboard.dashboard_id}"

# Clickable link first
displayHTML(f'<p><a href="{dashboard_url}" target="_blank" style="font-size:15px;">Open Contact Center QA Dashboard</a></p>')

print("=" * 70)
print("  DEPLOYED DASHBOARD")
print("=" * 70)
print()
print(f"  Name:     {DASHBOARD_NAME}")
print(f"  ID:       {dashboard.dashboard_id}")
print(f"  Path:     {DASHBOARD_PARENT_PATH}/{DASHBOARD_NAME}")
print(f"  Datasets: 5 (KPIs, Agent Leaderboard, Outlier Calls, Queue Perf, Coaching)")
print(f"  Export:   contact_center_qa.lvdash.json (for DABs CI/CD)")
print()
print("=" * 70)
print("  Next notebook: 05_Genie_AI_Skill")
print("  Creates the natural language interface + reusable AI Skill (run 03 first).")
print("=" * 70)

Open Contact Center QA Dashboard

  DEPLOYED DASHBOARD

  Name:     Contact Center QA Dashboard
  ID:       01f16e29bbaf1e3084e41b21dfa0f430
  Path:     /Workspace/Users/may.merkletan@databricks.com/Contact Center QA Dashboard
  Datasets: 5 (KPIs, Agent Leaderboard, Outlier Calls, Queue Perf, Coaching)
  Export:   contact_center_qa.lvdash.json (for DABs CI/CD)

  Next notebook: 05_Genie_AI_Skill
  Creates the natural language interface + reusable AI Skill (run 03 first).
